# K-Fold Cross-Validation for Gun Violence Prediction

This notebook implements k-fold cross-validation for the gun violence prediction pipeline, compares it with the original train-test split method, and analyzes the results.

---

**Outline:**
1. Import Required Libraries
2. Load and Prepare Dataset
3. Implement K-Fold Cross-Validation
4. Train and Evaluate Model with K-Fold
5. Compare with Existing Method
6. Display and Analyze Results

In [ ]:
# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 2. Load and Prepare Dataset
Load the cleaned dataset, handle missing values, and prepare features and target variable for modeling.

In [ ]:
# Load the preprocessed dataset (VIF-cleaned)
df = pd.read_csv('Gun_Violence_VIF_Clean.csv', index_col=None)

# Print all columns and their types for debugging
print('All columns and dtypes:')
print(df.dtypes)

# Explicitly drop GRID_ID if present
if 'GRID_ID' in df.columns:
    df = df.drop(columns=['GRID_ID'])

# Print again after dropping
print('Columns after dropping GRID_ID:')
print(df.dtypes)

# Only use numeric columns for features
X = df.drop(columns=['Total_Gun_Violence'])
X = X.loc[:, X.apply(lambda s: pd.api.types.is_numeric_dtype(s))]
y = (df['Total_Gun_Violence'] > 0).astype(int)

print('Columns in X:', list(X.columns))
print('Any object columns:', X.select_dtypes(include=['object']).columns.tolist())
print('First few rows of X:')
display(X.head())

# Display info
display(df.head())
print(f"Dataset shape: {df.shape}")
print(f"Features used for modeling: {list(X.columns)}")
print(f"Class balance: {np.bincount(y)} (No violence, Any violence)")

## 3. Implement K-Fold Cross-Validation
Set up stratified k-fold cross-validation to ensure class balance in each fold.

In [ ]:
# Set up stratified k-fold
k = 5
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

# Define models to evaluate
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(random_state=42)
}

# Metrics to collect
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

## 4. Train and Evaluate Model with K-Fold
Train each model using k-fold cross-validation, collect metrics for each fold, and compute the average and standard deviation.

In [ ]:
# Store results
kfold_results = {model: {m: [] for m in metrics} for model in models}

for model_name, model in models.items():
    print(f'Evaluating {model_name}...')
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
        kfold_results[model_name]['accuracy'].append(accuracy_score(y_test, y_pred))
        kfold_results[model_name]['precision'].append(precision_score(y_test, y_pred, zero_division=0))
        kfold_results[model_name]['recall'].append(recall_score(y_test, y_pred, zero_division=0))
        kfold_results[model_name]['f1'].append(f1_score(y_test, y_pred, zero_division=0))
        if y_prob is not None:
            kfold_results[model_name]['roc_auc'].append(roc_auc_score(y_test, y_prob))
        else:
            kfold_results[model_name]['roc_auc'].append(np.nan)
        print(f'  Fold {fold}: Acc={kfold_results[model_name]["accuracy"][-1]:.4f}  F1={kfold_results[model_name]["f1"][-1]:.4f}')

# Summarize results
kfold_summary = []
for model_name in models:
    row = {'Model': model_name}
    for m in metrics:
        scores = kfold_results[model_name][m]
        row[f'{m}_mean'] = np.nanmean(scores)
        row[f'{m}_std'] = np.nanstd(scores)
    kfold_summary.append(row)
kfold_summary_df = pd.DataFrame(kfold_summary)
display(kfold_summary_df)

# Print KFold means explicitly -- visible in any environment
print('=' * 70)
print('5-FOLD STRATIFIED CV  --  MEAN RESULTS')
print('=' * 70)
print('{:<22} {:>10} {:>10} {:>10} {:>8} {:>10}'.format('Model','Accuracy','Precision','Recall','F1','ROC-AUC'))
print('-' * 70)
for model_name in models:
    acc  = np.nanmean(kfold_results[model_name]['accuracy'])
    prec = np.nanmean(kfold_results[model_name]['precision'])
    rec  = np.nanmean(kfold_results[model_name]['recall'])
    f1v  = np.nanmean(kfold_results[model_name]['f1'])
    auc  = np.nanmean(kfold_results[model_name]['roc_auc'])
    print('{:<22} {:>10.4f} {:>10.4f} {:>10.4f} {:>8.4f} {:>10.4f}'.format(model_name, acc, prec, rec, f1v, auc))
print('=' * 70)

## 5. Compare with Existing Method
Run the original train-test split method and collect its performance metrics for comparison.

In [ ]:
# Train-test split (original method)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

split_results = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    row = {'Model': model_name}
    row['accuracy'] = accuracy_score(y_test, y_pred)
    row['precision'] = precision_score(y_test, y_pred, zero_division=0)
    row['recall'] = recall_score(y_test, y_pred, zero_division=0)
    row['f1'] = f1_score(y_test, y_pred, zero_division=0)
    row['roc_auc'] = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan
    split_results.append(row)
split_results_df = pd.DataFrame(split_results)
display(split_results_df)

## 6. Display and Analyze Results
Visualize and compare the results from k-fold cross-validation and the original train-test split. Discuss the differences in performance.

In [ ]:
# Rebuild kfold summary directly from kfold_results (bypasses any stale kfold_summary_df)
kfold_rows = []
for model_name in models:
    row = {'Model': model_name}
    for m in metrics:
        scores = kfold_results[model_name][m]
        row['kfold_' + m + '_mean'] = float(np.nanmean(scores))
        row['kfold_' + m + '_std']  = float(np.nanstd(scores))
    kfold_rows.append(row)
kfold_for_merge = pd.DataFrame(kfold_rows)

# Merge on Model column -- no index alignment issues
compare_df = split_results_df.merge(kfold_for_merge, on='Model')

# Show kfold_accuracy_mean prominently FIRST
kfold_acc_df = kfold_for_merge[['Model', 'kfold_accuracy_mean', 'kfold_accuracy_std',
                                 'kfold_f1_mean', 'kfold_f1_std',
                                 'kfold_roc_auc_mean', 'kfold_roc_auc_std']].copy()
kfold_acc_df.columns = ['Model', 'KFold Acc Mean', 'Acc Std',
                          'KFold F1 Mean', 'F1 Std',
                          'KFold ROC-AUC Mean', 'ROC-AUC Std']
print('=== K-Fold CV Key Metrics ===')
display(kfold_acc_df)

# Show full comparison table
print('')
print('=== Full Comparison (all metrics) ===')
display(compare_df)

# Plot F1 comparison
fig, ax = plt.subplots()
bar_width = 0.35
index = np.arange(len(compare_df))
ax.bar(index, compare_df['f1'], bar_width, label='Train-Test Split')
ax.bar(index + bar_width, compare_df['kfold_f1_mean'], bar_width,
       yerr=compare_df['kfold_f1_std'], label='K-Fold CV', alpha=0.7)
ax.set_xlabel('Model')
ax.set_ylabel('F1 Score')
ax.set_title('F1 Score Comparison: Train-Test Split vs. K-Fold CV')
ax.set_xticks(index + bar_width / 2)
ax.set_xticklabels(compare_df['Model'], rotation=30)
ax.legend()
plt.tight_layout()
plt.show()

# Print full metric comparison
print('=== Final Results: Train-Test Split vs K-Fold CV ===')
print('')
for _, row in compare_df.iterrows():
    print('Model: ' + row['Model'])
    for m in metrics:
        print('  {:<12}  Split: {:.4f}  |  K-Fold Mean: {:.4f} +/- {:.4f}'.format(
            m, row[m], row['kfold_' + m + '_mean'], row['kfold_' + m + '_std']))
    print('')

# Compact KFold accuracy summary
print('=' * 70)
print('KFOLD ACCURACY SUMMARY  (kfold_accuracy_mean)')
print('=' * 70)
print('{:<22} {:>10} {:>16} {:>8}'.format('Model', 'Split Acc', 'KFold Acc Mean', 'StdDev'))
print('-' * 60)
for _, row in compare_df.iterrows():
    print('{:<22} {:>10.4f} {:>16.4f} {:>8.4f}'.format(
        row['Model'], row['accuracy'], row['kfold_accuracy_mean'], row['kfold_accuracy_std']))
print('=' * 70)